In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/final_project_geoai"
INPUT_DIR = os.path.join(BASE_DIR, "inputs")

STACK_MAY = os.path.join(INPUT_DIR, "stack_may.tif")
STACK_JULY = os.path.join(INPUT_DIR, "stack_july.tif")
STACK_SEP = os.path.join(INPUT_DIR, "stack_sep.tif")

EXTENT_MASK = os.path.join(INPUT_DIR, "extent_mask.tif")
BOUNDARY_MASK = os.path.join(INPUT_DIR, "boundary_mask.tif")

!pip install rasterio segmentation-models-pytorch --quiet

Inputs alignment verification

In [ ]:
import rasterio

files = [
    STACK_MAY,
    STACK_JULY,
    STACK_SEP,
    EXTENT_MASK,
    BOUNDARY_MASK
]

for f in files:
    with rasterio.open(f) as src:
        print(os.path.basename(f))
        print("Shape:", src.count, src.height, src.width)
        print("Transform:", src.transform)
        print("-"*40)

In [ ]:
import rasterio
import numpy as np

PATCH_SIZE = 256
STRIDE = 128   # no overlap
MIN_BOUNDARY_PIXELS = 20

Rasters

In [ ]:
with rasterio.open(STACK_MAY) as src:
    may = src.read()

with rasterio.open(STACK_JULY) as src:
    july = src.read()

with rasterio.open(STACK_SEP) as src:
    sep = src.read()

with rasterio.open(EXTENT_MASK) as src:
    extent = src.read(1)

with rasterio.open(BOUNDARY_MASK) as src:
    boundary = src.read(1)

Patches extraction

In [ ]:
X = []
Y = []

for i in range(0, extent.shape[0] - PATCH_SIZE, STRIDE):
    for j in range(0, extent.shape[1] - PATCH_SIZE, STRIDE):

        b_patch = boundary[i:i+PATCH_SIZE, j:j+PATCH_SIZE]

        if b_patch.sum() < MIN_BOUNDARY_PIXELS:
            continue

        img_patch = may[:, i:i+PATCH_SIZE, j:j+PATCH_SIZE]
        label_patch = extent[i:i+PATCH_SIZE, j:j+PATCH_SIZE]

        X.append(img_patch)
        Y.append(label_patch)

X = np.array(X)
Y = np.array(Y)

print("Patches extracted:", len(X))
print("X shape:", X.shape)
print("Y shape:", Y.shape)

Train / Val / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, Y_train, Y_temp = train_test_split(
    X, Y, test_size=0.30, random_state=42
)

X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp, test_size=0.50, random_state=42
)

print(X_train.shape, X_val.shape, X_test.shape)

Normalization

In [ ]:
mean = X_train.mean(axis=(0,2,3), keepdims=True)
std  = X_train.std(axis=(0,2,3), keepdims=True)

X_train = (X_train - mean) / std
X_val   = (X_val - mean) / std
X_test  = (X_test - mean) / std

PyTorch tensors

In [ ]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

Y_train = torch.tensor(Y_train, dtype=torch.float32).unsqueeze(1)
Y_val   = torch.tensor(Y_val, dtype=torch.float32).unsqueeze(1)
Y_test  = torch.tensor(Y_test, dtype=torch.float32).unsqueeze(1)

DataLoaders

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(
    TensorDataset(X_train, Y_train),
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, Y_val),
    batch_size=8,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test, Y_test),
    batch_size=8,
    shuffle=False
)

U-Net Model

In [ ]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=6,
    classes=1
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(device)

Loss + Optimizer

In [ ]:
import torch.nn as nn

pos_weight = torch.tensor([8.0]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

Training loops (10 epochs)

In [ ]:
EPOCHS = 40

train_losses = []
val_losses = []

for epoch in range(EPOCHS):

    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)

            preds = model(xb)
            loss = criterion(preds, yb)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}"
    )

In [ ]:
import os
import matplotlib.pyplot as plt

# Ensure save folder exists
save_dir = "/content/drive/MyDrive/GeoAI_Final_Project/visualizations"
os.makedirs(save_dir, exist_ok=True)

plt.figure(figsize=(8,5))
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()

save_path = os.path.join(save_dir, "loss_curve.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')

print(f"Saved to: {save_path}")

plt.show()

Metrics

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, jaccard_score, precision_score, recall_score

model.eval()

all_preds = []
all_truth = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)

        preds = torch.sigmoid(model(xb)).cpu().numpy()
        preds_bin = (preds > 0.5).astype(np.uint8)

        all_preds.append(preds_bin)
        all_truth.append(yb.numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_truth = np.concatenate(all_truth, axis=0)

# Flatten
y_pred = all_preds.flatten()
y_true = all_truth.flatten()

f1 = f1_score(y_true, y_pred)
iou = jaccard_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

print(f"F1 Score:   {f1:.4f}")
print(f"IoU:        {iou:.4f}")
print(f"Precision:  {precision:.4f}")
print(f"Recall:     {recall:.4f}")

Visualization

In [ ]:
import os
import matplotlib.pyplot as plt
import torch

save_dir = "/content/drive/MyDrive/GeoAI_Final_Project/visualizations"
os.makedirs(save_dir, exist_ok=True)

model.eval()

xb, yb = next(iter(test_loader))
xb = xb.to(device)

with torch.no_grad():
    preds = torch.sigmoid(model(xb)).cpu().numpy()

xb = xb.cpu().numpy()
yb = yb.numpy()

sample_scores = yb[:,0].sum(axis=(1,2))
sample_indices = sample_scores.argsort()[-3:][::-1]

for i, idx in enumerate(sample_indices):

    fig = plt.figure(figsize=(15,5))

    plt.subplot(1,3,1)
    plt.imshow(xb[idx,3], cmap='gray')
    plt.title("Input (Band 8)")

    plt.subplot(1,3,2)
    plt.imshow(yb[idx,0], cmap='gray')
    plt.title("Ground Truth")

    plt.subplot(1,3,3)
    plt.imshow(preds[idx,0] > 0.5, cmap='gray')
    plt.title("Prediction")

    plt.tight_layout()

    fig.savefig(
        os.path.join(save_dir, f"prediction_sample_{i+1}.png"),
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()
    plt.close()